---
## 7. Elaboração da Busca em Grade (Grid Search)

Conforme solicitado no trabalho pedagógico, foi elaborada uma busca em grade focando em:

- **Parâmetro de Regularização (C)**: Controla o trade-off entre margem e erros
- **Funções de Kernel**: RBF, Linear e Polinomial
- **Gamma**: Coeficiente do kernel RBF e Polinomial

### Justificativa dos Valores

| Parâmetro | Valores | Justificativa |
|-----------|---------|---------------|
| C | 0.1, 1, 10, 100, 1000 | Faixa logarítmica para diferentes níveis de regularização |
| Kernel | rbf, linear, poly | Separação não-linear, linear e polinomial |
| Gamma | scale, auto, 0.01, 0.001 | Valores padrão e exploração manual |
| Degree | 2, 3, 4 | Graus típicos para kernel polinomial |

In [77]:
# Grade de hiperparâmetros expandida
param_grid_svm = {
    'C': [0.1, 1, 10, 100, 1000],
    'kernel': ['rbf', 'linear', 'poly'],
    'gamma': ['scale', 'auto', 0.01, 0.001, 0.0001],
    'degree': [2, 3, 4]  # para kernel poly
}

print("\n" + "="*70)
print("GRADE DE HIPERPARÂMETROS PARA SVM")
print("="*70)
print(f"\n C (Regularização):     {param_grid_svm['C']}")
print(f" Kernel:                {param_grid_svm['kernel']}")
print(f" Gamma:                 {param_grid_svm['gamma']}")
print(f" Degree (poly):         {param_grid_svm['degree']}")
print("="*70)


GRADE DE HIPERPARÂMETROS PARA SVM

 C (Regularização):     [0.1, 1, 10, 100, 1000]
 Kernel:                ['rbf', 'linear', 'poly']
 Gamma:                 ['scale', 'auto', 0.01, 0.001, 0.0001]
 Degree (poly):         [2, 3, 4]


In [78]:
# Grid Search em diferentes configurações de features
print("\n" + "="*70)
print(" GRID SEARCH - MÚLTIPLAS CONFIGURAÇÕES")
print("="*70)

configurations = [
    ('Fused (Inception+VGG16+EfficientNet)', X_train_fused_scaled, X_test_fused_scaled),
    (f'Fused + PCA({best_n_components})', X_train_pca_final, X_test_pca_final),
]

all_results = []

for config_name, X_train_cfg, X_test_cfg in configurations:
    print(f"\n{'='*70}")
    print(f" Configuração: {config_name}")
    print(f"   Shape: {X_train_cfg.shape}")
    print(f"{'='*70}")

    # Grid Search
    svm = SVC(random_state=RANDOM_STATE)
    grid_search = GridSearchCV(
        estimator=svm,
        param_grid=param_grid_svm,
        cv=5,
        scoring='accuracy',
        n_jobs=-1,
        verbose=1
    )

    print(f"\n Iniciando Grid Search...")
    start_time = time.time()
    grid_search.fit(X_train_cfg, y_train)
    training_time = time.time() - start_time

    # Melhor modelo
    best_model = grid_search.best_estimator_
    y_pred = best_model.predict(X_test_cfg)

    # Métricas
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, average='weighted', zero_division=0)
    recall = recall_score(y_test, y_pred, average='weighted', zero_division=0)
    f1 = f1_score(y_test, y_pred, average='weighted', zero_division=0)

    print(f"\n Melhores Hiperparâmetros:")
    for param, value in grid_search.best_params_.items():
        print(f"   {param}: {value}")

    print(f"\n Resultados:")
    print(f"   CV Score: {grid_search.best_score_:.4f}")
    print(f"   Acurácia: {accuracy:.4f} ({accuracy*100:.2f}%)")
    print(f"   F1-Score: {f1:.4f}")
    print(f"   Tempo: {training_time/60:.1f} min")

    all_results.append({
        'Config': config_name,
        'Best_Params': grid_search.best_params_,
        'CV_Score': grid_search.best_score_,
        'Accuracy': accuracy,
        'Precision': precision,
        'Recall': recall,
        'F1': f1,
        'Time': training_time,
        'Model': best_model,
        'y_pred': y_pred,
        'X_train': X_train_cfg,
        'X_test': X_test_cfg
    })


 GRID SEARCH - MÚLTIPLAS CONFIGURAÇÕES

 Configuração: Fused (Inception+VGG16+EfficientNet)
   Shape: (1120, 3840)

 Iniciando Grid Search...
Fitting 5 folds for each of 225 candidates, totalling 1125 fits

 Melhores Hiperparâmetros:
   C: 10
   degree: 2
   gamma: 0.0001
   kernel: rbf

 Resultados:
   CV Score: 0.7580
   Acurácia: 0.7571 (75.71%)
   F1-Score: 0.7612
   Tempo: 8.6 min

 Configuração: Fused + PCA(700)
   Shape: (1120, 700)

 Iniciando Grid Search...
Fitting 5 folds for each of 225 candidates, totalling 1125 fits

 Melhores Hiperparâmetros:
   C: 0.1
   degree: 2
   gamma: scale
   kernel: linear

 Resultados:
   CV Score: 0.7527
   Acurácia: 0.7500 (75.00%)
   F1-Score: 0.7546
   Tempo: 1.2 min


8. Ensemble Methods — Meta-Learning e Voting

Após a identificação da melhor configuração de features (combinação de CNNs com maior desempenho), diferentes classificadores foram treinados de forma independente utilizando exatamente o mesmo conjunto de dados. O objetivo dessa etapa é explorar a diversidade de decisões entre modelos, uma vez que classificadores distintos podem capturar padrões complementares a partir das mesmas representações visuais.

Foram utilizados classificadores base com diferentes vieses indutivos, incluindo variações do SVM (kernels RBF, polinomial e linear) e Random Forest. Cada modelo foi avaliado individualmente no conjunto de teste, permitindo comparar seus desempenhos e selecionar candidatos para estratégias de ensemble, como voting e stacking.

Essa abordagem segue o princípio de que a combinação de múltiplos classificadores pode resultar em maior robustez e melhor capacidade de generalização, especialmente em problemas de classificação multiclasse com alta complexidade visual.


---



In [79]:
# Treinar múltiplos classificadores para ensemble
print("\n" + "="*70)
print(" TREINANDO CLASSIFICADORES PARA ENSEMBLE")
print("="*70)

# Usar a melhor configuração de features
best_config = max(all_results, key=lambda x: x['Accuracy'])
X_train_best = best_config['X_train']
X_test_best = best_config['X_test']

print(f"\nUsando configuração: {best_config['Config']}")
print(f"Shape: {X_train_best.shape}")

# Classificadores base
classificadores = [
    ('SVM_RBF', SVC(kernel='rbf', C=10, gamma='scale', probability=True, random_state=RANDOM_STATE)),
    ('SVM_Polinomial', SVC(kernel='poly', C=10, degree=3, probability=True, random_state=RANDOM_STATE)),
    ('SVM_Linear', SVC(kernel='linear', C=1, probability=True, random_state=RANDOM_STATE)),
    ('RandomForest', RandomForestClassifier(n_estimators=200, max_depth=20, n_jobs=-1, random_state=RANDOM_STATE)),
]

resultados_classificadores = []

print("\n Treinando classificadores individuais...\n")
for nome, clf in classificadores:
    inicio = time.time()
    clf.fit(X_train_best, y_train)
    y_pred = clf.predict(X_test_best)
    acc = accuracy_score(y_test, y_pred)
    tempo = time.time() - inicio

    resultados_classificadores.append({
        'nome': nome,
        'modelo': clf,
        'acuracia': acc,
        'tempo': tempo
    })

    print(f"   {nome:<20} Acuracia: {acc:.4f} ({acc*100:.2f}%)  Tempo: {tempo:.1f}s")

# Ordenar por acurácia
resultados_classificadores = sorted(resultados_classificadores, key=lambda x: x['acuracia'], reverse=True)


 TREINANDO CLASSIFICADORES PARA ENSEMBLE

Usando configuração: Fused (Inception+VGG16+EfficientNet)
Shape: (1120, 3840)

 Treinando classificadores individuais...

   SVM_RBF              Acuracia: 0.7536 (75.36%)  Tempo: 21.8s
   SVM_Polinomial       Acuracia: 0.5571 (55.71%)  Tempo: 19.3s
   SVM_Linear           Acuracia: 0.7429 (74.29%)  Tempo: 10.6s
   RandomForest         Acuracia: 0.7000 (70.00%)  Tempo: 1.8s


### Stacking (Meta-Learning)

Nesta etapa, é aplicada a técnica de *Stacking*, uma estratégia de ensemble baseada em meta-aprendizado. Diferentemente de métodos simples de votação, o stacking aprende automaticamente como combinar as predições dos classificadores base por meio de um modelo de nível superior (meta-classificador).

Foram selecionados os três classificadores com maior desempenho individual na etapa anterior, garantindo que apenas modelos competitivos contribuíssem para o ensemble. Esses classificadores atuam como estimadores base, produzindo predições que são utilizadas como entrada para o meta-classificador.

Como meta-classificador, foi utilizada a Regressão Logística, escolhida por sua simplicidade, estabilidade e boa capacidade de generalização. O treinamento do stacking foi realizado com validação cruzada, reduzindo o risco de overfitting e promovendo uma combinação mais robusta das decisões individuais.

O objetivo desta abordagem é capturar padrões complementares entre os modelos base e verificar se a combinação aprendida supera o desempenho dos classificadores isolados e das CNNs individuais, conforme observado no artigo de referência.


In [80]:
# Stacking - Meta-classificador
print("\n" + "="*70)
print(" STACKING (META-LEARNING)")
print("="*70)

# Usar os 3 melhores classificadores como base
top3_classifiers = [(r['name'], r['model']) for r in classifier_results[:3]]

print(f"\n Base estimators (Top 3):")
for name, _ in top3_classifiers:
    print(f"   • {name}")

# Stacking com Logistic Regression como meta-classifier
stacking_clf = StackingClassifier(
    estimators=top3_classifiers,
    final_estimator=LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
    cv=5,
    n_jobs=-1
)

print("\n Treinando Stacking...")
start_time = time.time()
stacking_clf.fit(X_train_best, y_train)
y_pred_stacking = stacking_clf.predict(X_test_best)
acc_stacking = accuracy_score(y_test, y_pred_stacking)
stacking_time = time.time() - start_time

print(f"\n Stacking Accuracy: {acc_stacking:.4f} ({acc_stacking*100:.2f}%)")
print(f" Tempo: {stacking_time:.1f}s")


 STACKING (META-LEARNING)

 Base estimators (Top 3):
   • SVM_RBF
   • SVM_Poly
   • SVM_Linear

 Treinando Stacking...

 Stacking Accuracy: 0.7714 (77.14%)
 Tempo: 54.7s


### Soft Voting Ensemble

Além do stacking, foi implementado um ensemble baseado em *Soft Voting*, no qual as probabilidades preditas por cada classificador base são combinadas por meio de uma média ponderada implícita.

Diferentemente do hard voting, que considera apenas as classes finais, o soft voting utiliza as probabilidades estimadas, permitindo decisões mais informadas, especialmente em cenários de incerteza entre classes visualmente semelhantes.

Foram utilizados classificadores heterogêneos, incluindo variações do SVM com diferentes kernels e um Random Forest, promovendo diversidade de decisões no ensemble. Cada modelo foi treinado novamente a partir da melhor configuração de features identificada anteriormente, garantindo consistência experimental.

O desempenho do soft voting é comparado tanto aos modelos individuais quanto ao stacking, permitindo avaliar empiricamente qual estratégia de ensemble apresenta melhor capacidade de generalização para o problema de classificação multiclasse das sementes de cacau.


In [81]:
# Soft Voting Ensemble
print("\n" + "="*70)
print(" SOFT VOTING ENSEMBLE")
print("="*70)

# Criar novo ensemble com classificadores novos (para evitar refit)
voting_estimators = [
    ('SVM_RBF', SVC(kernel='rbf', C=10, gamma='scale', probability=True, random_state=RANDOM_STATE)),
    ('SVM_Poly', SVC(kernel='poly', C=10, degree=3, probability=True, random_state=RANDOM_STATE)),
    ('RandomForest', RandomForestClassifier(n_estimators=200, max_depth=20, random_state=RANDOM_STATE)),
]

voting_clf = VotingClassifier(
    estimators=voting_estimators,
    voting='soft',
    n_jobs=-1
)

print("\n Treinando Soft Voting...")
start_time = time.time()
voting_clf.fit(X_train_best, y_train)
y_pred_voting = voting_clf.predict(X_test_best)
acc_voting = accuracy_score(y_test, y_pred_voting)
voting_time = time.time() - start_time

print(f"\n Soft Voting Accuracy: {acc_voting:.4f} ({acc_voting*100:.2f}%)")
print(f" Tempo: {voting_time:.1f}s")


 SOFT VOTING ENSEMBLE

 Treinando Soft Voting...

 Soft Voting Accuracy: 0.7643 (76.43%)
 Tempo: 22.4s


### Considerações sobre Ensemble Methods

Os resultados obtidos com stacking e soft voting indicam que a combinação de múltiplos classificadores pode produzir ganhos adicionais de desempenho em relação aos modelos individuais. Essa evidência está alinhada com a literatura, que aponta ensembles como uma estratégia eficaz para problemas complexos de visão computacional, especialmente quando há variabilidade visual significativa entre classes.

Essas abordagens complementam a análise baseada em CNNs individuais, contribuindo para uma avaliação mais abrangente do potencial do Transfer Learning aplicado ao teste de corte de sementes de cacau.


---
## 9. Análise dos Resultados